# SQL Definitions

In [ ]:
SQL_SCRIPTS = {}

## 00_staging

In [ ]:
SQL_SCRIPTS['00_staging'] = r"""
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS intermediate;
CREATE SCHEMA IF NOT EXISTS marts;
CREATE SCHEMA IF NOT EXISTS analysis;

CREATE OR REPLACE TABLE staging.stg_customers AS
SELECT
    customer_id::VARCHAR AS customer_id,
    customer_unique_id::VARCHAR AS customer_unique_id,
    TRY_CAST(customer_zip_code_prefix AS INTEGER) AS customer_zip_code_prefix,
    customer_city::VARCHAR AS customer_city,
    customer_state::VARCHAR AS customer_state,
    'olist_customers_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto('{{RAW_DIR}}/olist_customers_dataset.csv', header = true, sample_size = -1);

CREATE OR REPLACE TABLE staging.stg_geolocation AS
SELECT
    TRY_CAST(geolocation_zip_code_prefix AS INTEGER) AS geolocation_zip_code_prefix,
    TRY_CAST(geolocation_lat AS DOUBLE) AS geolocation_lat,
    TRY_CAST(geolocation_lng AS DOUBLE) AS geolocation_lng,
    geolocation_city::VARCHAR AS geolocation_city,
    geolocation_state::VARCHAR AS geolocation_state,
    'olist_geolocation_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto('{{RAW_DIR}}/olist_geolocation_dataset.csv', header = true, sample_size = -1);

CREATE OR REPLACE TABLE staging.stg_order_items AS
SELECT
    order_id::VARCHAR AS order_id,
    TRY_CAST(order_item_id AS INTEGER) AS order_item_id,
    product_id::VARCHAR AS product_id,
    seller_id::VARCHAR AS seller_id,
    TRY_CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
    TRY_CAST(price AS DECIMAL(18, 2)) AS price,
    TRY_CAST(freight_value AS DECIMAL(18, 2)) AS freight_value,
    'olist_order_items_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto('{{RAW_DIR}}/olist_order_items_dataset.csv', header = true, sample_size = -1);

CREATE OR REPLACE TABLE staging.stg_order_payments AS
SELECT
    order_id::VARCHAR AS order_id,
    TRY_CAST(payment_sequential AS INTEGER) AS payment_sequential,
    payment_type::VARCHAR AS payment_type,
    TRY_CAST(payment_installments AS INTEGER) AS payment_installments,
    TRY_CAST(payment_value AS DECIMAL(18, 2)) AS payment_value,
    'olist_order_payments_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto('{{RAW_DIR}}/olist_order_payments_dataset.csv', header = true, sample_size = -1);

CREATE OR REPLACE TABLE staging.stg_order_reviews AS
SELECT
    review_id::VARCHAR AS review_id,
    order_id::VARCHAR AS order_id,
    TRY_CAST(review_score AS INTEGER) AS review_score,
    review_comment_title::VARCHAR AS review_comment_title,
    review_comment_message::VARCHAR AS review_comment_message,
    TRY_CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
    TRY_CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp,
    'olist_order_reviews_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto(
    '{{RAW_DIR}}/olist_order_reviews_dataset.csv',
    header = true,
    sample_size = -1,
    -- strict_mode=false tolerates malformed free-text review comments in the public file.
    -- TRY_CAST converts invalid review scores and timestamps to NULL; has_valid_review
    -- excludes missing or out-of-range scores from review metrics.
    strict_mode = false
);

CREATE OR REPLACE TABLE staging.stg_orders AS
SELECT
    order_id::VARCHAR AS order_id,
    customer_id::VARCHAR AS customer_id,
    order_status::VARCHAR AS order_status,
    TRY_CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
    TRY_CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
    TRY_CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
    TRY_CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
    TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date,
    'olist_orders_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto('{{RAW_DIR}}/olist_orders_dataset.csv', header = true, sample_size = -1);

CREATE OR REPLACE TABLE staging.stg_products AS
SELECT
    product_id::VARCHAR AS product_id,
    product_category_name::VARCHAR AS product_category_name,
    TRY_CAST(product_name_lenght AS INTEGER) AS product_name_length,
    TRY_CAST(product_description_lenght AS INTEGER) AS product_description_length,
    TRY_CAST(product_photos_qty AS INTEGER) AS product_photos_qty,
    TRY_CAST(product_weight_g AS DOUBLE) AS product_weight_g,
    TRY_CAST(product_length_cm AS DOUBLE) AS product_length_cm,
    TRY_CAST(product_height_cm AS DOUBLE) AS product_height_cm,
    TRY_CAST(product_width_cm AS DOUBLE) AS product_width_cm,
    'olist_products_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto('{{RAW_DIR}}/olist_products_dataset.csv', header = true, sample_size = -1);

CREATE OR REPLACE TABLE staging.stg_sellers AS
SELECT
    seller_id::VARCHAR AS seller_id,
    TRY_CAST(seller_zip_code_prefix AS INTEGER) AS seller_zip_code_prefix,
    seller_city::VARCHAR AS seller_city,
    seller_state::VARCHAR AS seller_state,
    'olist_sellers_dataset.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto('{{RAW_DIR}}/olist_sellers_dataset.csv', header = true, sample_size = -1);

CREATE OR REPLACE TABLE staging.stg_category_translation AS
SELECT
    product_category_name::VARCHAR AS product_category_name,
    product_category_name_english::VARCHAR AS product_category_name_english,
    'product_category_name_translation.csv'::VARCHAR AS source_file,
    CURRENT_TIMESTAMP AS ingested_at
FROM read_csv_auto(
    '{{RAW_DIR}}/product_category_name_translation.csv',
    header = true,
    sample_size = -1
);

CREATE OR REPLACE TABLE staging.source_control_totals AS
SELECT * FROM (
    VALUES
        ('stg_customers', 99441, (SELECT COUNT(*) FROM staging.stg_customers)),
        ('stg_geolocation', 1000163, (SELECT COUNT(*) FROM staging.stg_geolocation)),
        ('stg_order_items', 112650, (SELECT COUNT(*) FROM staging.stg_order_items)),
        ('stg_order_payments', 103886, (SELECT COUNT(*) FROM staging.stg_order_payments)),
        ('stg_order_reviews', 99224, (SELECT COUNT(*) FROM staging.stg_order_reviews)),
        ('stg_orders', 99441, (SELECT COUNT(*) FROM staging.stg_orders)),
        ('stg_products', 32951, (SELECT COUNT(*) FROM staging.stg_products)),
        ('stg_sellers', 3095, (SELECT COUNT(*) FROM staging.stg_sellers)),
        ('stg_category_translation', 71, (SELECT COUNT(*) FROM staging.stg_category_translation))
) AS t(table_name, expected_row_count, actual_row_count);

"""

## 10_intermediate

In [ ]:
SQL_SCRIPTS['10_intermediate'] = r"""
CREATE OR REPLACE TABLE intermediate.int_zip_geolocation AS
SELECT
    geolocation_zip_code_prefix AS zip_code_prefix,
    MEDIAN(geolocation_lat) AS latitude,
    MEDIAN(geolocation_lng) AS longitude,
    COUNT(*) AS source_coordinate_count
FROM staging.stg_geolocation
WHERE geolocation_zip_code_prefix IS NOT NULL
  AND geolocation_lat BETWEEN -90 AND 90
  AND geolocation_lng BETWEEN -180 AND 180
GROUP BY geolocation_zip_code_prefix;

CREATE OR REPLACE TABLE intermediate.int_order_payments AS
SELECT
    order_id,
    COUNT(*) AS payment_row_count,
    COUNT(DISTINCT payment_type) AS payment_type_count,
    SUM(payment_value) AS payment_total,
    MAX(payment_installments) AS maximum_installments
FROM staging.stg_order_payments
GROUP BY order_id;

CREATE OR REPLACE TABLE intermediate.int_order_reviews AS
WITH ranked AS (
    SELECT
        order_id,
        review_id,
        review_score,
        review_creation_date,
        review_answer_timestamp,
        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY
                review_answer_timestamp DESC NULLS LAST,
                review_creation_date DESC NULLS LAST,
                review_id DESC
        ) AS representative_rank,
        COUNT(*) OVER (PARTITION BY order_id) AS review_row_count,
        AVG(review_score) OVER (PARTITION BY order_id) AS average_all_review_rows
    FROM staging.stg_order_reviews
)
SELECT
    order_id,
    review_row_count,
    average_all_review_rows,
    review_id AS representative_review_id,
    review_score AS representative_review_score,
    review_creation_date AS representative_review_creation_date,
    review_answer_timestamp AS representative_review_answer_timestamp
FROM ranked
WHERE representative_rank = 1;

CREATE OR REPLACE TABLE intermediate.int_order_items AS
SELECT
    oi.order_id,
    COUNT(*) AS item_count,
    COUNT(DISTINCT oi.product_id) AS distinct_product_count,
    COUNT(DISTINCT oi.seller_id) AS seller_count,
    COUNT(DISTINCT p.product_category_name) AS known_category_count,
    SUM(oi.price) AS gross_item_value,
    SUM(oi.freight_value) AS freight_value,
    SUM(oi.price + oi.freight_value) AS item_plus_freight_value
FROM staging.stg_order_items AS oi
LEFT JOIN staging.stg_products AS p USING (product_id)
GROUP BY oi.order_id;

CREATE OR REPLACE TABLE intermediate.int_order_seller_items AS
WITH item_sla AS (
    SELECT
        oi.order_id,
        oi.seller_id,
        oi.order_item_id,
        oi.shipping_limit_date,
        oi.price,
        oi.freight_value,
        o.order_delivered_carrier_date,
        CASE
            WHEN o.order_delivered_carrier_date IS NOT NULL
             AND oi.shipping_limit_date IS NOT NULL
            THEN EPOCH(o.order_delivered_carrier_date - oi.shipping_limit_date) / 3600.0
        END AS item_handover_variance_hours,
        CASE
            WHEN o.order_delivered_carrier_date IS NOT NULL
             AND oi.shipping_limit_date IS NOT NULL
            THEN o.order_delivered_carrier_date > oi.shipping_limit_date
        END AS item_handover_sla_breach
    FROM staging.stg_order_items AS oi
    LEFT JOIN staging.stg_orders AS o USING (order_id)
)
SELECT
    order_id,
    seller_id,
    COUNT(*) AS seller_item_count,
    SUM(price) AS seller_gross_item_value,
    SUM(freight_value) AS seller_freight_value,
    MIN(shipping_limit_date) AS minimum_shipping_limit_date,
    MAX(shipping_limit_date) AS maximum_shipping_limit_date,
    COUNT(shipping_limit_date) AS shipping_limit_nonnull_count,
    COUNT(DISTINCT shipping_limit_date) AS distinct_shipping_limit_count,
    COUNT(*) FILTER (WHERE item_handover_sla_breach) AS breached_item_count,
    BOOL_OR(item_handover_sla_breach) AS any_item_handover_breach,
    BOOL_AND(item_handover_sla_breach) FILTER (
        WHERE item_handover_sla_breach IS NOT NULL
    ) AS all_items_handover_breach,
    MAX(item_handover_variance_hours) AS maximum_handover_variance_hours,
    MAX(GREATEST(item_handover_variance_hours, 0)) AS maximum_positive_handover_delay_hours
FROM item_sla
GROUP BY order_id, seller_id;

CREATE OR REPLACE TABLE intermediate.int_order_categories AS
SELECT
    oi.order_id,
    COALESCE(
        NULLIF(t.product_category_name_english, ''),
        NULLIF(p.product_category_name, ''),
        'unknown'
    ) AS category_name,
    COUNT(*) AS category_item_count,
    SUM(oi.price) AS category_gross_item_value,
    SUM(oi.freight_value) AS category_freight_value
FROM staging.stg_order_items AS oi
LEFT JOIN staging.stg_products AS p USING (product_id)
LEFT JOIN staging.stg_category_translation AS t USING (product_category_name)
GROUP BY oi.order_id, category_name;

"""

## 20_marts

In [ ]:
SQL_SCRIPTS['20_marts'] = r"""
CREATE OR REPLACE TABLE marts.mart_order_fulfillment AS
WITH joined AS (
    SELECT
        o.order_id,
        o.customer_id,
        c.customer_unique_id,
        o.order_status,
        o.order_purchase_timestamp,
        DATE_TRUNC('month', o.order_purchase_timestamp)::DATE AS purchase_month,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        c.customer_zip_code_prefix,
        c.customer_city,
        c.customer_state,
        oi.item_count,
        oi.distinct_product_count,
        oi.seller_count,
        oi.known_category_count,
        oi.gross_item_value,
        oi.freight_value,
        oi.item_plus_freight_value,
        pay.payment_row_count,
        pay.payment_type_count,
        pay.payment_total,
        pay.maximum_installments,
        rev.review_row_count,
        rev.average_all_review_rows,
        rev.representative_review_id,
        rev.representative_review_score,
        rev.representative_review_creation_date,
        rev.representative_review_answer_timestamp,
        geo.latitude AS customer_latitude,
        geo.longitude AS customer_longitude,
        o.order_purchase_timestamp >= TIMESTAMP '2017-01-01'
            AND o.order_purchase_timestamp < TIMESTAMP '2018-09-01'
            AS is_primary_analysis_period,
        o.order_approved_at IS NOT NULL
            AND o.order_purchase_timestamp IS NOT NULL
            AND o.order_approved_at >= o.order_purchase_timestamp
            AS is_approval_duration_valid,
        o.order_delivered_carrier_date IS NOT NULL
            AND o.order_approved_at IS NOT NULL
            AND o.order_delivered_carrier_date >= o.order_approved_at
            AS is_precarrier_duration_valid,
        o.order_delivered_customer_date IS NOT NULL
            AND o.order_delivered_carrier_date IS NOT NULL
            AND o.order_delivered_customer_date >= o.order_delivered_carrier_date
            AS is_carrier_duration_valid,
        o.order_delivered_customer_date IS NOT NULL
            AND o.order_purchase_timestamp IS NOT NULL
            AND o.order_delivered_customer_date >= o.order_purchase_timestamp
            AS is_end_to_end_duration_valid,
        o.order_status = 'delivered'
            AND o.order_delivered_customer_date IS NOT NULL
            AND o.order_estimated_delivery_date IS NOT NULL
            AND o.order_purchase_timestamp IS NOT NULL
            AND o.order_delivered_customer_date >= o.order_purchase_timestamp
            AS is_delivery_sla_eligible,
        rev.representative_review_score BETWEEN 1 AND 5 AS has_valid_review,
        geo.latitude IS NOT NULL AND geo.longitude IS NOT NULL AS has_valid_customer_geolocation
    FROM staging.stg_orders AS o
    LEFT JOIN staging.stg_customers AS c USING (customer_id)
    LEFT JOIN intermediate.int_order_items AS oi USING (order_id)
    LEFT JOIN intermediate.int_order_payments AS pay USING (order_id)
    LEFT JOIN intermediate.int_order_reviews AS rev USING (order_id)
    LEFT JOIN intermediate.int_zip_geolocation AS geo
        ON c.customer_zip_code_prefix = geo.zip_code_prefix
)
SELECT
    *,
    CASE
        WHEN is_delivery_sla_eligible
        THEN order_delivered_customer_date::DATE <= order_estimated_delivery_date::DATE
    END AS on_time_delivery,
    CASE
        WHEN is_delivery_sla_eligible
        THEN DATE_DIFF(
            'day',
            order_estimated_delivery_date::DATE,
            order_delivered_customer_date::DATE
        )
    END AS delivery_variance_days,
    CASE
        WHEN is_approval_duration_valid
        THEN EPOCH(order_approved_at - order_purchase_timestamp) / 3600.0
    END AS approval_lead_time_hours,
    CASE
        WHEN is_precarrier_duration_valid
        THEN EPOCH(order_delivered_carrier_date - order_approved_at) / 3600.0
    END AS precarrier_handover_time_hours,
    CASE
        WHEN is_carrier_duration_valid
        THEN EPOCH(order_delivered_customer_date - order_delivered_carrier_date) / 3600.0
    END AS carrier_delivery_time_hours,
    CASE
        WHEN is_end_to_end_duration_valid
        THEN EPOCH(order_delivered_customer_date - order_purchase_timestamp) / 3600.0
    END AS end_to_end_fulfillment_time_hours
FROM joined;

CREATE OR REPLACE TABLE marts.mart_order_seller AS
WITH seller_joined AS (
    SELECT
        osi.order_id,
        osi.seller_id,
        mof.purchase_month,
        mof.order_status,
        mof.order_purchase_timestamp,
        mof.order_delivered_carrier_date,
        mof.order_delivered_customer_date,
        mof.order_estimated_delivery_date,
        mof.is_primary_analysis_period,
        mof.is_delivery_sla_eligible,
        mof.on_time_delivery,
        mof.delivery_variance_days,
        mof.has_valid_review,
        mof.representative_review_score,
        mof.customer_state,
        mof.customer_zip_code_prefix,
        mof.customer_latitude,
        mof.customer_longitude,
        oi.seller_count,
        osi.seller_item_count,
        osi.seller_gross_item_value,
        osi.seller_freight_value,
        osi.minimum_shipping_limit_date,
        osi.maximum_shipping_limit_date,
        osi.shipping_limit_nonnull_count,
        osi.distinct_shipping_limit_count,
        osi.breached_item_count,
        osi.any_item_handover_breach,
        osi.all_items_handover_breach,
        osi.maximum_handover_variance_hours,
        osi.maximum_positive_handover_delay_hours,
        s.seller_zip_code_prefix,
        s.seller_city,
        s.seller_state,
        geo.latitude AS seller_latitude,
        geo.longitude AS seller_longitude
    FROM intermediate.int_order_seller_items AS osi
    INNER JOIN marts.mart_order_fulfillment AS mof USING (order_id)
    INNER JOIN intermediate.int_order_items AS oi USING (order_id)
    LEFT JOIN staging.stg_sellers AS s USING (seller_id)
    LEFT JOIN intermediate.int_zip_geolocation AS geo
        ON s.seller_zip_code_prefix = geo.zip_code_prefix
), flags AS (
    SELECT
        *,
        seller_count = 1 AS seller_attribution_reliable,
        is_primary_analysis_period
            AND seller_count = 1
            AND order_status NOT IN ('canceled', 'unavailable')
            AS is_handover_candidate,
        is_primary_analysis_period
            AND order_status NOT IN ('canceled', 'unavailable')
            AND order_delivered_carrier_date IS NOT NULL
            AND shipping_limit_nonnull_count = seller_item_count
            AS is_handover_sla_observable,
        is_primary_analysis_period
            AND seller_count = 1
            AND order_status NOT IN ('canceled', 'unavailable')
            AND order_delivered_carrier_date IS NOT NULL
            AND shipping_limit_nonnull_count = seller_item_count
            AS is_handover_sla_eligible,
        seller_latitude IS NOT NULL
            AND seller_longitude IS NOT NULL
            AND customer_latitude IS NOT NULL
            AND customer_longitude IS NOT NULL
            AS has_valid_distance
    FROM seller_joined
)
SELECT
    *,
    CASE
        WHEN is_handover_sla_observable THEN any_item_handover_breach
    END AS handover_sla_breach,
    CASE
        WHEN seller_state IS NOT NULL AND customer_state IS NOT NULL
        THEN seller_state = customer_state
    END AS is_within_state,
    CASE
        WHEN has_valid_distance THEN
            2 * 6371 * ASIN(
                LEAST(
                    1,
                    SQRT(
                        POW(SIN(RADIANS(customer_latitude - seller_latitude) / 2), 2)
                        + COS(RADIANS(seller_latitude))
                        * COS(RADIANS(customer_latitude))
                        * POW(SIN(RADIANS(customer_longitude - seller_longitude) / 2), 2)
                    )
                )
            )
    END AS straight_line_distance_km
FROM flags;

CREATE OR REPLACE TABLE marts.mart_order_category AS
SELECT
    category.order_id,
    category.category_name,
    category.category_item_count,
    category.category_gross_item_value,
    category.category_freight_value,
    orders.purchase_month,
    orders.order_status,
    orders.is_primary_analysis_period,
    orders.is_delivery_sla_eligible,
    orders.on_time_delivery,
    orders.delivery_variance_days,
    orders.is_approval_duration_valid,
    orders.is_precarrier_duration_valid,
    orders.is_carrier_duration_valid,
    orders.is_end_to_end_duration_valid,
    orders.approval_lead_time_hours,
    orders.precarrier_handover_time_hours,
    orders.carrier_delivery_time_hours,
    orders.end_to_end_fulfillment_time_hours,
    orders.has_valid_review,
    orders.representative_review_score,
    orders.customer_state
FROM intermediate.int_order_categories AS category
INNER JOIN marts.mart_order_fulfillment AS orders USING (order_id);

CREATE OR REPLACE TABLE marts.mart_monthly_fulfillment_kpi AS
WITH order_kpis AS (
    SELECT
        purchase_month,
        COUNT(*) AS total_order_count,
        COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered_order_count,
        COUNT(*) FILTER (WHERE order_status = 'canceled') AS canceled_order_count,
        COUNT(*) FILTER (WHERE order_status = 'unavailable') AS unavailable_order_count,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS delivery_sla_eligible_count,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND on_time_delivery) AS on_time_delivery_count,
        COUNT(*) FILTER (WHERE is_end_to_end_duration_valid) AS end_to_end_eligible_count,
        QUANTILE_CONT(end_to_end_fulfillment_time_hours, 0.5)
            FILTER (WHERE is_end_to_end_duration_valid) AS end_to_end_p50_hours,
        QUANTILE_CONT(end_to_end_fulfillment_time_hours, 0.9)
            FILTER (WHERE is_end_to_end_duration_valid) AS end_to_end_p90_hours,
        COUNT(*) FILTER (WHERE has_valid_review) AS reviewed_order_count,
        AVG(representative_review_score) FILTER (WHERE has_valid_review) AS average_review_score
    FROM marts.mart_order_fulfillment
    WHERE is_primary_analysis_period
    GROUP BY purchase_month
), handover_kpis AS (
    SELECT
        purchase_month,
        COUNT(*) FILTER (WHERE is_handover_candidate) AS handover_candidate_count,
        COUNT(*) FILTER (WHERE is_handover_sla_eligible) AS handover_sla_eligible_count,
        COUNT(*) FILTER (
            WHERE is_handover_sla_eligible AND handover_sla_breach
        ) AS handover_breach_count
    FROM marts.mart_order_seller
    WHERE is_primary_analysis_period AND seller_attribution_reliable
    GROUP BY purchase_month
)
SELECT
    o.*,
    o.total_order_count - o.delivery_sla_eligible_count AS delivery_sla_excluded_count,
    o.canceled_order_count::DOUBLE / NULLIF(o.total_order_count, 0) AS cancellation_rate,
    o.unavailable_order_count::DOUBLE / NULLIF(o.total_order_count, 0) AS unavailable_rate,
    o.on_time_delivery_count::DOUBLE / NULLIF(o.delivery_sla_eligible_count, 0)
        AS on_time_delivery_rate,
    o.reviewed_order_count::DOUBLE / NULLIF(o.total_order_count, 0) AS review_response_coverage,
    h.handover_candidate_count,
    h.handover_sla_eligible_count,
    h.handover_candidate_count - h.handover_sla_eligible_count AS handover_sla_excluded_count,
    h.handover_breach_count,
    h.handover_breach_count::DOUBLE / NULLIF(h.handover_sla_eligible_count, 0)
        AS handover_breach_rate,
    h.handover_sla_eligible_count::DOUBLE / NULLIF(h.handover_candidate_count, 0)
        AS handover_observation_coverage
FROM order_kpis AS o
LEFT JOIN handover_kpis AS h USING (purchase_month)
ORDER BY purchase_month;

CREATE OR REPLACE TABLE marts.mart_seller_monthly_sla AS
WITH seller_month AS (
    SELECT
        purchase_month,
        seller_id,
        COUNT(*) AS eligible_order_count,
        COUNT(*) FILTER (WHERE handover_sla_breach) AS handover_breach_count,
        AVG(maximum_handover_variance_hours) AS average_handover_variance_hours,
        QUANTILE_CONT(maximum_positive_handover_delay_hours, 0.5)
            FILTER (WHERE handover_sla_breach) AS breach_delay_p50_hours,
        QUANTILE_CONT(maximum_positive_handover_delay_hours, 0.9)
            FILTER (WHERE handover_sla_breach) AS breach_delay_p90_hours
    FROM marts.mart_order_seller
    WHERE is_handover_sla_eligible
    GROUP BY purchase_month, seller_id
)
SELECT
    sm.*,
    sm.handover_breach_count::DOUBLE / NULLIF(sm.eligible_order_count, 0)
        AS seller_handover_breach_rate,
    mk.handover_breach_rate AS platform_handover_breach_rate,
    sm.handover_breach_count::DOUBLE / NULLIF(sm.eligible_order_count, 0)
        - mk.handover_breach_rate AS breach_rate_gap
FROM seller_month AS sm
INNER JOIN marts.mart_monthly_fulfillment_kpi AS mk USING (purchase_month);

"""

## 30_data_checks

In [ ]:
SQL_SCRIPTS["30_data_checks"] = r"""
CREATE SCHEMA IF NOT EXISTS checks;

CREATE OR REPLACE VIEW checks.required_checks (
    check_id, check_name, table_name, rows_tested, affected_count, expected_result
) AS
SELECT
    'B001',
    'Source row counts match the dataset',
    'staging.source_control_totals',
    COUNT(*)::BIGINT,
    COUNT(*) FILTER (WHERE expected_row_count <> actual_row_count)::BIGINT,
    '0 mismatched tables'
FROM staging.source_control_totals

UNION ALL

SELECT
    'B002',
    'Order mart has one row per source order',
    'marts.mart_order_fulfillment',
    COUNT(*),
    ABS(COUNT(*) - 99441) + (COUNT(*) - COUNT(DISTINCT order_id)),
    '99441 rows and unique order_id'
FROM marts.mart_order_fulfillment

UNION ALL

SELECT
    'B003',
    'Order-seller mart grain is unique',
    'marts.mart_order_seller',
    COUNT(*),
    COUNT(*) - COUNT(DISTINCT (order_id, seller_id)),
    '0 duplicate order_id × seller_id rows'
FROM marts.mart_order_seller

UNION ALL

SELECT
    'B004',
    'Order-seller mart reconciles to source pairs',
    'marts.mart_order_seller',
    COUNT(*),
    ABS(
        COUNT(*) - (
            SELECT COUNT(*)
            FROM (SELECT DISTINCT order_id, seller_id FROM staging.stg_order_items)
        )
    ),
    'Matches distinct source order_id × seller_id pairs'
FROM marts.mart_order_seller

UNION ALL

SELECT
    'B005',
    'Order-category mart grain is unique',
    'marts.mart_order_category',
    COUNT(*),
    COUNT(*) - COUNT(DISTINCT (order_id, category_name)),
    '0 duplicate order_id × category rows'
FROM marts.mart_order_category

UNION ALL

SELECT
    'B006',
    'Gross item value reconciles',
    'intermediate.int_order_items',
    1,
    CASE WHEN ABS(
        (SELECT SUM(price) FROM staging.stg_order_items)
        - (SELECT SUM(gross_item_value) FROM intermediate.int_order_items)
    ) > 0.01 THEN 1 ELSE 0 END,
    'Absolute difference <= 0.01'

UNION ALL

SELECT
    'B007',
    'Freight value reconciles',
    'intermediate.int_order_items',
    1,
    CASE WHEN ABS(
        (SELECT SUM(freight_value) FROM staging.stg_order_items)
        - (SELECT SUM(freight_value) FROM intermediate.int_order_items)
    ) > 0.01 THEN 1 ELSE 0 END,
    'Absolute difference <= 0.01'

UNION ALL

SELECT
    'B008',
    'Payment total reconciles',
    'intermediate.int_order_payments',
    1,
    CASE WHEN ABS(
        (SELECT SUM(payment_value) FROM staging.stg_order_payments)
        - (SELECT SUM(payment_total) FROM intermediate.int_order_payments)
    ) > 0.01 THEN 1 ELSE 0 END,
    'Absolute difference <= 0.01'

UNION ALL

SELECT
    'B009',
    'KPI numerators do not exceed denominators',
    'marts.mart_monthly_fulfillment_kpi',
    COUNT(*),
    COUNT(*) FILTER (
        WHERE on_time_delivery_count > delivery_sla_eligible_count
           OR handover_breach_count > handover_sla_eligible_count
           OR delivered_order_count > total_order_count
    ),
    '0 invalid monthly rows'
FROM marts.mart_monthly_fulfillment_kpi

UNION ALL

SELECT
    'B010',
    'Monthly KPI denominators are nonzero',
    'marts.mart_monthly_fulfillment_kpi',
    COUNT(*),
    COUNT(*) FILTER (
        WHERE total_order_count = 0
           OR delivery_sla_eligible_count = 0
           OR handover_sla_eligible_count = 0
    ),
    '0 monthly rows with zero denominators'
FROM marts.mart_monthly_fulfillment_kpi

UNION ALL

SELECT
    'B011',
    'Critical staging keys and numeric fields parse successfully',
    'staging',
    (SELECT COUNT(*) FROM staging.stg_orders)
        + (SELECT COUNT(*) FROM staging.stg_order_items)
        + (SELECT COUNT(*) FROM staging.stg_order_payments),
    (SELECT COUNT(*) FROM staging.stg_orders
        WHERE order_id IS NULL OR customer_id IS NULL OR order_purchase_timestamp IS NULL)
        + (SELECT COUNT(*) FROM staging.stg_order_items
            WHERE order_id IS NULL OR order_item_id IS NULL OR product_id IS NULL
               OR seller_id IS NULL OR price IS NULL OR freight_value IS NULL)
        + (SELECT COUNT(*) FROM staging.stg_order_payments
            WHERE order_id IS NULL OR payment_value IS NULL),
    '0 critical parse failures';


CREATE OR REPLACE VIEW checks.data_notes (
    check_id, check_name, table_name, rows_tested, affected_count, handling
) AS
SELECT
    'W001',
    'Delivered orders with carrier handover before approval',
    'marts.mart_order_fulfillment',
    COUNT(*) FILTER (WHERE order_status = 'delivered'),
    COUNT(*) FILTER (
        WHERE order_status = 'delivered'
          AND order_delivered_carrier_date < order_approved_at
    ),
    'Flag and exclude from pre-carrier duration'
FROM marts.mart_order_fulfillment

UNION ALL

SELECT
    'W002',
    'Delivered orders with customer delivery before carrier handover',
    'marts.mart_order_fulfillment',
    COUNT(*) FILTER (WHERE order_status = 'delivered'),
    COUNT(*) FILTER (
        WHERE order_status = 'delivered'
          AND order_delivered_customer_date < order_delivered_carrier_date
    ),
    'Flag and exclude from carrier duration'
FROM marts.mart_order_fulfillment

UNION ALL

SELECT
    'W003',
    'Delivered orders missing customer-delivery timestamp',
    'marts.mart_order_fulfillment',
    COUNT(*) FILTER (WHERE order_status = 'delivered'),
    COUNT(*) FILTER (
        WHERE order_status = 'delivered' AND order_delivered_customer_date IS NULL
    ),
    'Exclude from delivery SLA'
FROM marts.mart_order_fulfillment

UNION ALL

SELECT
    'W004',
    'Orders containing multiple sellers',
    'intermediate.int_order_items',
    COUNT(*),
    COUNT(*) FILTER (WHERE seller_count > 1),
    'Exclude from primary seller attribution'
FROM intermediate.int_order_items

UNION ALL

SELECT
    'W005',
    'Orders containing multiple known categories',
    'intermediate.int_order_items',
    COUNT(*),
    COUNT(*) FILTER (WHERE known_category_count > 1),
    'Treat category counts as non-additive'
FROM intermediate.int_order_items

UNION ALL

SELECT
    'W006',
    'Orders containing multiple review rows',
    'intermediate.int_order_reviews',
    COUNT(*),
    COUNT(*) FILTER (WHERE review_row_count > 1),
    'Use deterministic representative review'
FROM intermediate.int_order_reviews

UNION ALL

SELECT
    'W007',
    'Orders missing valid customer ZIP-prefix coordinates',
    'marts.mart_order_fulfillment',
    COUNT(*),
    COUNT(*) FILTER (WHERE NOT has_valid_customer_geolocation),
    'Exclude only from distance analysis'
FROM marts.mart_order_fulfillment

UNION ALL

SELECT
    'W008',
    'Order-seller pairs with multiple shipping deadlines',
    'intermediate.int_order_seller_items',
    COUNT(*),
    COUNT(*) FILTER (WHERE distinct_shipping_limit_count > 1),
    'Preserve item-level deadline logic'
FROM intermediate.int_order_seller_items

UNION ALL

SELECT
    'W009',
    'Promised-day deliveries misclassified by timestamp comparison',
    'marts.mart_order_fulfillment',
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible),
    COUNT(*) FILTER (
        WHERE is_delivery_sla_eligible
          AND order_delivered_customer_date::DATE = order_estimated_delivery_date::DATE
          AND order_delivered_customer_date > order_estimated_delivery_date
    ),
    'Use calendar-date comparison'
FROM marts.mart_order_fulfillment

UNION ALL

SELECT
    'W010',
    'Order-seller pairs missing seller ZIP-prefix coordinates',
    'marts.mart_order_seller',
    COUNT(*),
    COUNT(*) FILTER (WHERE seller_latitude IS NULL OR seller_longitude IS NULL),
    'Exclude only from distance analysis'
FROM marts.mart_order_seller

UNION ALL

SELECT
    'W011',
    'Geolocation rows with invalid coordinate ranges',
    'staging.stg_geolocation',
    COUNT(*),
    COUNT(*) FILTER (
        WHERE geolocation_lat IS NULL OR geolocation_lng IS NULL
           OR geolocation_lat NOT BETWEEN -90 AND 90
           OR geolocation_lng NOT BETWEEN -180 AND 180
    ),
    'Exclude invalid coordinates from ZIP representative points'
FROM staging.stg_geolocation

UNION ALL

SELECT
    'W012',
    'Order items missing usable category enrichment',
    'staging.stg_order_items',
    COUNT(*),
    COUNT(*) FILTER (
        WHERE COALESCE(
            NULLIF(translation.product_category_name_english, ''),
            NULLIF(products.product_category_name, '')
        ) IS NULL
    ),
    'Retain as unknown category'
FROM staging.stg_order_items AS items
LEFT JOIN staging.stg_products AS products USING (product_id)
LEFT JOIN staging.stg_category_translation AS translation USING (product_category_name)

UNION ALL

SELECT
    'W013',
    'Order payment differs from item plus freight value',
    'intermediate.int_order_payments',
    COUNT(*),
    COUNT(*) FILTER (WHERE ABS(payment.payment_total - items.item_plus_freight_value) > 0.01),
    'Keep payment and item-value concepts separate'
FROM intermediate.int_order_payments AS payment
INNER JOIN intermediate.int_order_items AS items USING (order_id)

UNION ALL

SELECT
    'W014',
    'Purchases in incomplete terminal months',
    'staging.stg_orders',
    COUNT(*),
    COUNT(*) FILTER (WHERE order_purchase_timestamp >= TIMESTAMP '2018-09-01'),
    'Exclude from the primary reporting cohort'
FROM staging.stg_orders

UNION ALL

SELECT
    'W015',
    'Canceled or unavailable orders without seller coverage',
    'marts.mart_order_fulfillment',
    COUNT(*) FILTER (
        WHERE fulfillment.order_status IN ('canceled', 'unavailable')
    ),
    COUNT(*) FILTER (
        WHERE fulfillment.order_status IN ('canceled', 'unavailable')
          AND items.seller_count IS NULL
    ),
    'Exclude seller cancellation attribution'
FROM marts.mart_order_fulfillment AS fulfillment
LEFT JOIN intermediate.int_order_items AS items USING (order_id);

"""

## 40_analysis

In [ ]:
analysis_sql = {}

### Seller priority criteria

In [ ]:
analysis_sql["parameters"] = r"""
CREATE OR REPLACE TABLE analysis.queue_parameters AS
SELECT * FROM (
    VALUES (
        30, 20, 5, 2, 5, 0.05, 2, 0.01, 0.05
    )
) AS p(
    minimum_seller_orders,
    watchlist_minimum_orders,
    minimum_seller_month_orders,
    minimum_active_months,
    minimum_breach_count,
    monthly_rate_gap,
    repeated_month_requirement,
    high_contribution_share,
    customer_warning_gap
);
"""

### Time spent in each delivery stage

In [ ]:
analysis_sql["stage_times"] = r"""
CREATE OR REPLACE TABLE analysis.fulfillment_stage_summary AS
SELECT
    stage_name,
    eligible_order_count,
    excluded_order_count,
    p50_hours,
    p75_hours,
    p90_hours
FROM (
    SELECT
        'Purchase to approval' AS stage_name,
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND is_approval_duration_valid)
            AS eligible_order_count,
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND NOT is_approval_duration_valid)
            AS excluded_order_count,
        QUANTILE_CONT(approval_lead_time_hours, 0.5)
            FILTER (WHERE is_primary_analysis_period AND is_approval_duration_valid) AS p50_hours,
        QUANTILE_CONT(approval_lead_time_hours, 0.75)
            FILTER (WHERE is_primary_analysis_period AND is_approval_duration_valid) AS p75_hours,
        QUANTILE_CONT(approval_lead_time_hours, 0.9)
            FILTER (WHERE is_primary_analysis_period AND is_approval_duration_valid) AS p90_hours
    FROM marts.mart_order_fulfillment

    UNION ALL
    SELECT
        'Approval to carrier handover',
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND is_precarrier_duration_valid),
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND NOT is_precarrier_duration_valid),
        QUANTILE_CONT(precarrier_handover_time_hours, 0.5)
            FILTER (WHERE is_primary_analysis_period AND is_precarrier_duration_valid),
        QUANTILE_CONT(precarrier_handover_time_hours, 0.75)
            FILTER (WHERE is_primary_analysis_period AND is_precarrier_duration_valid),
        QUANTILE_CONT(precarrier_handover_time_hours, 0.9)
            FILTER (WHERE is_primary_analysis_period AND is_precarrier_duration_valid)
    FROM marts.mart_order_fulfillment

    UNION ALL
    SELECT
        'Carrier handover to customer delivery',
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND is_carrier_duration_valid),
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND NOT is_carrier_duration_valid),
        QUANTILE_CONT(carrier_delivery_time_hours, 0.5)
            FILTER (WHERE is_primary_analysis_period AND is_carrier_duration_valid),
        QUANTILE_CONT(carrier_delivery_time_hours, 0.75)
            FILTER (WHERE is_primary_analysis_period AND is_carrier_duration_valid),
        QUANTILE_CONT(carrier_delivery_time_hours, 0.9)
            FILTER (WHERE is_primary_analysis_period AND is_carrier_duration_valid)
    FROM marts.mart_order_fulfillment

    UNION ALL
    SELECT
        'Purchase to customer delivery',
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND is_end_to_end_duration_valid),
        COUNT(*) FILTER (WHERE is_primary_analysis_period AND NOT is_end_to_end_duration_valid),
        QUANTILE_CONT(end_to_end_fulfillment_time_hours, 0.5)
            FILTER (WHERE is_primary_analysis_period AND is_end_to_end_duration_valid),
        QUANTILE_CONT(end_to_end_fulfillment_time_hours, 0.75)
            FILTER (WHERE is_primary_analysis_period AND is_end_to_end_duration_valid),
        QUANTILE_CONT(end_to_end_fulfillment_time_hours, 0.9)
            FILTER (WHERE is_primary_analysis_period AND is_end_to_end_duration_valid)
    FROM marts.mart_order_fulfillment
) AS stages;
"""

### Late handover and late delivery

In [ ]:
analysis_sql["handover_delivery"] = r"""
CREATE OR REPLACE TABLE analysis.handover_delivery_failure_modes AS
SELECT
    CASE
        WHEN NOT handover_sla_breach AND on_time_delivery THEN 'Handover met / Delivery on time'
        WHEN handover_sla_breach AND on_time_delivery THEN 'Handover breached / Delivery on time'
        WHEN NOT handover_sla_breach AND NOT on_time_delivery THEN 'Handover met / Delivery late'
        WHEN handover_sla_breach AND NOT on_time_delivery THEN 'Handover breached / Delivery late'
    END AS failure_mode,
    COUNT(*) AS order_count,
    AVG(representative_review_score) FILTER (WHERE has_valid_review) AS average_review_score,
    COUNT(*) FILTER (WHERE has_valid_review) AS reviewed_order_count
FROM marts.mart_order_seller
WHERE is_handover_sla_eligible
  AND is_delivery_sla_eligible
GROUP BY failure_mode;
"""

### Delivery delay and review scores

In [ ]:
analysis_sql["reviews"] = r"""
CREATE OR REPLACE TABLE analysis.delivery_review_association AS
SELECT
    CASE
        WHEN on_time_delivery THEN 'On time'
        WHEN delivery_variance_days BETWEEN 1 AND 3 THEN 'Late 1-3 days'
        WHEN delivery_variance_days BETWEEN 4 AND 7 THEN 'Late 4-7 days'
        WHEN delivery_variance_days > 7 THEN 'Late 8+ days'
    END AS delivery_group,
    COUNT(*) AS delivery_sla_eligible_count,
    COUNT(*) FILTER (WHERE has_valid_review) AS reviewed_order_count,
    COUNT(*) FILTER (WHERE has_valid_review)::DOUBLE / COUNT(*) AS review_response_coverage,
    AVG(representative_review_score) FILTER (WHERE has_valid_review) AS average_review_score,
    COUNT(*) FILTER (WHERE has_valid_review AND representative_review_score <= 2)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE has_valid_review), 0) AS low_review_rate
FROM marts.mart_order_fulfillment
WHERE is_primary_analysis_period AND is_delivery_sla_eligible
GROUP BY delivery_group;
"""

### Delivery performance by product category

In [ ]:
analysis_sql["categories"] = r"""
CREATE OR REPLACE TABLE analysis.agg_category_performance AS
WITH platform AS (
    SELECT
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)::DOUBLE
            / NULLIF(COUNT(*) FILTER (WHERE is_delivery_sla_eligible), 0)
            AS platform_late_delivery_rate,
        COUNT(*) FILTER (
            WHERE has_valid_review AND representative_review_score <= 2
        )::DOUBLE / NULLIF(COUNT(*) FILTER (WHERE has_valid_review), 0)
            AS platform_low_review_rate
    FROM marts.mart_order_fulfillment
    WHERE is_primary_analysis_period
), category_rollup AS (
    SELECT
        category_name,
        COUNT(*) AS category_involved_order_count,
        SUM(category_item_count) AS category_item_count,
        SUM(category_gross_item_value) AS category_gross_item_value,
        SUM(category_freight_value) AS category_freight_value,
        COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered_order_count,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS delivery_sla_eligible_count,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND on_time_delivery)
            AS on_time_delivery_count,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)
            AS late_delivery_order_count,
        AVG(delivery_variance_days) FILTER (WHERE is_delivery_sla_eligible)
            AS average_delivery_variance_days,
        AVG(delivery_variance_days) FILTER (
            WHERE is_delivery_sla_eligible AND NOT on_time_delivery
        ) AS average_late_days,
        QUANTILE_CONT(delivery_variance_days, 0.9) FILTER (
            WHERE is_delivery_sla_eligible AND NOT on_time_delivery
        ) AS p90_late_days,
        COUNT(*) FILTER (WHERE is_end_to_end_duration_valid) AS end_to_end_eligible_count,
        QUANTILE_CONT(end_to_end_fulfillment_time_hours, 0.5)
            FILTER (WHERE is_end_to_end_duration_valid) AS end_to_end_p50_hours,
        QUANTILE_CONT(end_to_end_fulfillment_time_hours, 0.9)
            FILTER (WHERE is_end_to_end_duration_valid) AS end_to_end_p90_hours,
        COUNT(*) FILTER (WHERE has_valid_review) AS reviewed_order_count,
        AVG(representative_review_score) FILTER (WHERE has_valid_review)
            AS average_review_score,
        COUNT(*) FILTER (
            WHERE has_valid_review AND representative_review_score <= 2
        ) AS low_review_order_count
    FROM marts.mart_order_category
    WHERE is_primary_analysis_period
    GROUP BY category_name
)
SELECT
    category.*,
    category.category_involved_order_count - category.delivery_sla_eligible_count
        AS delivery_sla_excluded_count,
    category.delivery_sla_eligible_count::DOUBLE
        / NULLIF(category.category_involved_order_count, 0) AS delivery_sla_coverage,
    category.on_time_delivery_count::DOUBLE
        / NULLIF(category.delivery_sla_eligible_count, 0) AS on_time_delivery_rate,
    category.late_delivery_order_count::DOUBLE
        / NULLIF(category.delivery_sla_eligible_count, 0) AS late_delivery_rate,
    category.late_delivery_order_count::DOUBLE
        / NULLIF(SUM(category.late_delivery_order_count) OVER (), 0)
        AS share_of_category_involved_late_records,
    platform.platform_late_delivery_rate,
    category.late_delivery_order_count::DOUBLE
        / NULLIF(category.delivery_sla_eligible_count, 0)
        - platform.platform_late_delivery_rate AS late_delivery_rate_gap,
    category.reviewed_order_count::DOUBLE
        / NULLIF(category.category_involved_order_count, 0) AS review_response_coverage,
    category.low_review_order_count::DOUBLE
        / NULLIF(category.reviewed_order_count, 0) AS low_review_rate,
    platform.platform_low_review_rate,
    category.low_review_order_count::DOUBLE
        / NULLIF(category.reviewed_order_count, 0)
        - platform.platform_low_review_rate AS low_review_rate_gap
FROM category_rollup AS category
CROSS JOIN platform;

CREATE OR REPLACE TABLE analysis.category_fulfillment_stage_performance AS
WITH category_stage_rows AS (
    SELECT
        category_name,
        'Purchase to approval' AS stage_name,
        is_approval_duration_valid AS is_stage_valid,
        approval_lead_time_hours AS stage_hours
    FROM marts.mart_order_category
    WHERE is_primary_analysis_period

    UNION ALL
    SELECT
        category_name,
        'Approval to carrier handover',
        is_precarrier_duration_valid,
        precarrier_handover_time_hours
    FROM marts.mart_order_category
    WHERE is_primary_analysis_period

    UNION ALL
    SELECT
        category_name,
        'Carrier handover to customer delivery',
        is_carrier_duration_valid,
        carrier_delivery_time_hours
    FROM marts.mart_order_category
    WHERE is_primary_analysis_period

    UNION ALL
    SELECT
        category_name,
        'Purchase to customer delivery',
        is_end_to_end_duration_valid,
        end_to_end_fulfillment_time_hours
    FROM marts.mart_order_category
    WHERE is_primary_analysis_period
)
SELECT
    category_name,
    stage_name,
    COUNT(*) FILTER (WHERE is_stage_valid) AS eligible_order_count,
    COUNT(*) FILTER (WHERE NOT is_stage_valid) AS excluded_order_count,
    QUANTILE_CONT(stage_hours, 0.5) FILTER (WHERE is_stage_valid) AS p50_hours,
    QUANTILE_CONT(stage_hours, 0.75) FILTER (WHERE is_stage_valid) AS p75_hours,
    QUANTILE_CONT(stage_hours, 0.9) FILTER (WHERE is_stage_valid) AS p90_hours
FROM category_stage_rows
GROUP BY category_name, stage_name;
"""

### Delivery performance by state pair

In [ ]:
analysis_sql["state_pairs"] = r"""
CREATE OR REPLACE TABLE analysis.agg_state_lane_performance AS
SELECT
    seller_state,
    customer_state,
    seller_state = customer_state AS is_within_state,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS eligible_delivery_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)
        AS late_delivery_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE is_delivery_sla_eligible), 0) AS late_delivery_rate,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible) AS handover_eligible_order_count,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)
        AS handover_breach_count,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE is_handover_sla_eligible), 0) AS handover_breach_rate,
    AVG(straight_line_distance_km) FILTER (WHERE has_valid_distance)
        AS average_straight_line_distance_km,
    COUNT(*) FILTER (WHERE has_valid_distance) AS valid_distance_count
FROM marts.mart_order_seller
WHERE is_primary_analysis_period
  AND seller_attribution_reliable
  AND seller_state IS NOT NULL
  AND customer_state IS NOT NULL
GROUP BY seller_state, customer_state;
"""

### Seller performance and priority tiers

In [ ]:
analysis_sql["sellers"] = r"""
CREATE OR REPLACE TABLE analysis.agg_seller_performance AS
WITH platform AS (
    SELECT
        COUNT(*) AS platform_eligible_orders,
        COUNT(*) FILTER (WHERE handover_sla_breach) AS platform_breaches,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS platform_delivery_eligible,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)
            AS platform_late_deliveries,
        COUNT(*) FILTER (WHERE has_valid_review) AS platform_reviewed,
        COUNT(*) FILTER (WHERE has_valid_review AND representative_review_score <= 2)
            AS platform_low_reviews,
        QUANTILE_CONT(maximum_positive_handover_delay_hours, 0.9)
            FILTER (WHERE handover_sla_breach)
            AS platform_breach_delay_p90_hours
    FROM marts.mart_order_seller
    WHERE is_handover_sla_eligible
), monthly_repeats AS (
    SELECT
        seller_id,
        COUNT(*) FILTER (
            WHERE eligible_order_count >= 5 AND breach_rate_gap >= 0.05
        ) AS repeated_breach_month_count,
        COUNT(*) FILTER (WHERE eligible_order_count >= 5) AS volume_eligible_month_count
    FROM marts.mart_seller_monthly_sla
    GROUP BY seller_id
), seller AS (
    SELECT
        seller_id,
        MAX(seller_state) AS seller_state,
        COUNT(*) AS eligible_order_count,
        COUNT(DISTINCT purchase_month) AS eligible_active_month_count,
        COUNT(*) FILTER (WHERE handover_sla_breach) AS handover_breach_count,
        COUNT(*) FILTER (WHERE handover_sla_breach)::DOUBLE / COUNT(*)
            AS handover_breach_rate,
        QUANTILE_CONT(maximum_positive_handover_delay_hours, 0.5)
            FILTER (WHERE handover_sla_breach) AS breach_delay_p50_hours,
        QUANTILE_CONT(maximum_positive_handover_delay_hours, 0.9)
            FILTER (WHERE handover_sla_breach) AS breach_delay_p90_hours,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS delivery_eligible_order_count,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)
            AS late_delivery_order_count,
        COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)::DOUBLE
            / NULLIF(COUNT(*) FILTER (WHERE is_delivery_sla_eligible), 0)
            AS contextual_late_delivery_rate,
        COUNT(*) FILTER (WHERE has_valid_review) AS reviewed_order_count,
        COUNT(*) FILTER (WHERE has_valid_review AND representative_review_score <= 2)
            AS low_review_order_count,
        COUNT(*) FILTER (WHERE has_valid_review AND representative_review_score <= 2)::DOUBLE
            / NULLIF(COUNT(*) FILTER (WHERE has_valid_review), 0) AS contextual_low_review_rate
    FROM marts.mart_order_seller
    WHERE is_handover_sla_eligible
    GROUP BY seller_id
)
SELECT
    seller.*,
    seller.handover_breach_count::DOUBLE / NULLIF(platform.platform_breaches, 0)
        AS breach_contribution_share,
    monthly_repeats.volume_eligible_month_count,
    monthly_repeats.repeated_breach_month_count,
    platform.platform_breaches::DOUBLE / NULLIF(platform.platform_eligible_orders, 0)
        AS platform_handover_breach_rate,
    platform.platform_late_deliveries::DOUBLE / NULLIF(platform.platform_delivery_eligible, 0)
        AS platform_late_delivery_rate,
    platform.platform_low_reviews::DOUBLE / NULLIF(platform.platform_reviewed, 0)
        AS platform_low_review_rate,
    platform.platform_breach_delay_p90_hours
FROM seller
CROSS JOIN platform
LEFT JOIN monthly_repeats USING (seller_id);

CREATE OR REPLACE TABLE analysis.mart_seller_investigation_queue AS
WITH classified AS (
    SELECT
        seller.*,
        parameters.*,
        seller.eligible_order_count >= parameters.minimum_seller_orders
            AND seller.volume_eligible_month_count >= parameters.minimum_active_months
            AS has_sufficient_volume,
        seller.handover_breach_count >= parameters.minimum_breach_count
            AND seller.handover_breach_rate
                >= seller.platform_handover_breach_rate + parameters.monthly_rate_gap
            AS has_high_breach_rate,
        seller.breach_contribution_share >= parameters.high_contribution_share
            AS has_high_breach_contribution,
        seller.repeated_breach_month_count >= parameters.repeated_month_requirement
            AS has_repeated_monthly_breach,
        seller.breach_delay_p90_hours > seller.platform_breach_delay_p90_hours
            AS has_high_delay_severity,
        COALESCE(
            seller.contextual_late_delivery_rate
                > seller.platform_late_delivery_rate + parameters.customer_warning_gap,
            false
        ) OR COALESCE(
            seller.contextual_low_review_rate
                > seller.platform_low_review_rate + parameters.customer_warning_gap,
            false
        ) AS has_customer_outcome_warning,
        seller.eligible_order_count >= parameters.watchlist_minimum_orders
            AND seller.eligible_order_count < parameters.minimum_seller_orders
            AND seller.handover_breach_count >= parameters.minimum_breach_count
            AND seller.handover_breach_rate
                >= seller.platform_handover_breach_rate + parameters.monthly_rate_gap
            AS is_low_volume_watchlist
    FROM analysis.agg_seller_performance AS seller
    CROSS JOIN analysis.queue_parameters AS parameters
), prioritized AS (
    SELECT
        *,
        CASE
            WHEN has_sufficient_volume
             AND handover_breach_count >= minimum_breach_count
             AND has_repeated_monthly_breach
             AND (has_high_breach_contribution OR has_high_delay_severity)
            THEN 'P1'
            WHEN has_sufficient_volume
             AND handover_breach_count >= minimum_breach_count
             AND (has_high_breach_rate OR has_repeated_monthly_breach)
            THEN 'P2'
            WHEN has_sufficient_volume
             AND handover_breach_count >= minimum_breach_count
            THEN 'P3'
            WHEN is_low_volume_watchlist THEN 'WATCHLIST'
        END AS priority_tier
    FROM classified
)
SELECT
    seller_id,
    seller_state,
    priority_tier,
    CASE priority_tier
        WHEN 'P1' THEN 1
        WHEN 'P2' THEN 2
        WHEN 'P3' THEN 3
        WHEN 'WATCHLIST' THEN 4
    END AS priority_sort,
    eligible_order_count,
    eligible_active_month_count,
    handover_breach_count,
    handover_breach_rate,
    platform_handover_breach_rate,
    breach_contribution_share,
    volume_eligible_month_count,
    repeated_breach_month_count,
    breach_delay_p50_hours,
    breach_delay_p90_hours,
    platform_breach_delay_p90_hours,
    delivery_eligible_order_count,
    late_delivery_order_count,
    contextual_late_delivery_rate,
    reviewed_order_count,
    low_review_order_count,
    contextual_low_review_rate,
    CONCAT_WS(
        '|',
        CASE WHEN has_sufficient_volume THEN 'SUFFICIENT_VOLUME' END,
        CASE WHEN has_high_breach_rate THEN 'HIGH_BREACH_RATE' END,
        CASE WHEN has_high_breach_contribution THEN 'HIGH_BREACH_CONTRIBUTION' END,
        CASE WHEN has_repeated_monthly_breach THEN 'REPEATED_MONTHLY_BREACH' END,
        CASE WHEN has_high_delay_severity THEN 'HIGH_DELAY_SEVERITY' END,
        CASE WHEN has_customer_outcome_warning THEN 'CUSTOMER_OUTCOME_WARNING' END,
        CASE WHEN is_low_volume_watchlist THEN 'LOW_VOLUME_WATCHLIST' END
    ) AS reason_codes
FROM prioritized
WHERE priority_tier IS NOT NULL;
"""

### Seller counts under different thresholds

In [ ]:
analysis_sql["thresholds"] = r"""
CREATE OR REPLACE TABLE analysis.queue_threshold_sensitivity AS
WITH combinations AS (
    SELECT *
    FROM (VALUES (20), (30), (50)) AS orders(minimum_orders)
    CROSS JOIN (VALUES (0.03), (0.05), (0.10)) AS gaps(monthly_gap)
    CROSS JOIN (VALUES (2), (3)) AS repeats(repeated_month_requirement)
), seller_combinations AS (
    SELECT
        combinations.*,
        performance.seller_id,
        performance.eligible_order_count,
        performance.volume_eligible_month_count,
        performance.handover_breach_count,
        COUNT(*) FILTER (
            WHERE monthly.eligible_order_count >= (SELECT minimum_seller_month_orders FROM analysis.queue_parameters)
              AND monthly.breach_rate_gap >= combinations.monthly_gap
        ) AS repeated_breach_month_count
    FROM combinations
    CROSS JOIN analysis.agg_seller_performance AS performance
    LEFT JOIN marts.mart_seller_monthly_sla AS monthly
        ON performance.seller_id = monthly.seller_id
    GROUP BY
        combinations.minimum_orders,
        combinations.monthly_gap,
        combinations.repeated_month_requirement,
        performance.seller_id,
        performance.eligible_order_count,
        performance.volume_eligible_month_count,
        performance.handover_breach_count
)
SELECT
    minimum_orders,
    monthly_gap * 100 AS monthly_gap_pp,
    repeated_month_requirement,
    COUNT(*) FILTER (
        WHERE eligible_order_count >= minimum_orders
          AND volume_eligible_month_count >= (SELECT minimum_active_months FROM analysis.queue_parameters)
          AND handover_breach_count >= (SELECT minimum_breach_count FROM analysis.queue_parameters)
          AND repeated_breach_month_count >= repeated_month_requirement
    ) AS qualifying_seller_count
FROM seller_combinations
GROUP BY minimum_orders, monthly_gap, repeated_month_requirement;
"""

### Handover results by order group

In [ ]:
analysis_sql["handover_population"] = r"""
CREATE OR REPLACE TABLE analysis.handover_population_sensitivity AS
WITH populations AS (
    SELECT
        'PRIMARY_SINGLE_SELLER' AS population_name,
        'order_seller' AS population_grain,
        true AS seller_attribution_reliable,
        COUNT(*) FILTER (WHERE is_handover_candidate) AS candidate_count,
        COUNT(*) FILTER (WHERE is_handover_sla_eligible) AS eligible_count,
        COUNT(*) FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)
            AS breach_count,
        'Main analysis group' AS interpretation
    FROM marts.mart_order_seller

    UNION ALL
    SELECT
        'DELIVERED_ONLY_SINGLE_SELLER',
        'order_seller',
        true,
        COUNT(*) FILTER (
            WHERE is_primary_analysis_period
              AND seller_attribution_reliable
              AND order_status = 'delivered'
        ),
        COUNT(*) FILTER (
            WHERE is_handover_sla_eligible AND order_status = 'delivered'
        ),
        COUNT(*) FILTER (
            WHERE is_handover_sla_eligible
              AND order_status = 'delivered'
              AND handover_sla_breach
        ),
        'Sensitivity view restricted to delivered orders'
    FROM marts.mart_order_seller

    UNION ALL
    SELECT
        'MULTI_SELLER_OBSERVABLE',
        'order_seller',
        false,
        COUNT(*) FILTER (
            WHERE is_primary_analysis_period
              AND NOT seller_attribution_reliable
              AND order_status NOT IN ('canceled', 'unavailable')
        ),
        COUNT(*) FILTER (
            WHERE NOT seller_attribution_reliable AND is_handover_sla_observable
        ),
        COUNT(*) FILTER (
            WHERE NOT seller_attribution_reliable
              AND is_handover_sla_observable
              AND handover_sla_breach
        ),
        'Multi-seller orders; individual seller attribution unavailable'
    FROM marts.mart_order_seller
)
SELECT
    *,
    candidate_count - eligible_count AS excluded_count,
    breach_count::DOUBLE / NULLIF(eligible_count, 0) AS breach_rate,
    eligible_count::DOUBLE / NULLIF(candidate_count, 0) AS observation_coverage
FROM populations;
"""

### Stage times for late and on-time orders

In [ ]:
analysis_sql["stage_comparisons"] = r"""
CREATE OR REPLACE TABLE analysis.fulfillment_stage_by_delivery_outcome AS
WITH stage_rows AS (
    SELECT
        CASE WHEN on_time_delivery THEN 'On time' ELSE 'Late' END AS delivery_outcome,
        'Purchase to approval' AS stage_name,
        is_approval_duration_valid AS is_stage_valid,
        approval_lead_time_hours AS stage_hours
    FROM marts.mart_order_fulfillment
    WHERE is_primary_analysis_period AND is_delivery_sla_eligible
    UNION ALL
    SELECT
        CASE WHEN on_time_delivery THEN 'On time' ELSE 'Late' END,
        'Approval to carrier handover',
        is_precarrier_duration_valid,
        precarrier_handover_time_hours
    FROM marts.mart_order_fulfillment
    WHERE is_primary_analysis_period AND is_delivery_sla_eligible
    UNION ALL
    SELECT
        CASE WHEN on_time_delivery THEN 'On time' ELSE 'Late' END,
        'Carrier handover to customer delivery',
        is_carrier_duration_valid,
        carrier_delivery_time_hours
    FROM marts.mart_order_fulfillment
    WHERE is_primary_analysis_period AND is_delivery_sla_eligible
    UNION ALL
    SELECT
        CASE WHEN on_time_delivery THEN 'On time' ELSE 'Late' END,
        'Purchase to customer delivery',
        is_end_to_end_duration_valid,
        end_to_end_fulfillment_time_hours
    FROM marts.mart_order_fulfillment
    WHERE is_primary_analysis_period AND is_delivery_sla_eligible
)
SELECT
    delivery_outcome,
    stage_name,
    COUNT(*) FILTER (WHERE is_stage_valid) AS eligible_order_count,
    COUNT(*) FILTER (WHERE NOT is_stage_valid) AS excluded_order_count,
    QUANTILE_CONT(stage_hours, 0.5) FILTER (WHERE is_stage_valid) AS p50_hours,
    QUANTILE_CONT(stage_hours, 0.75) FILTER (WHERE is_stage_valid) AS p75_hours,
    QUANTILE_CONT(stage_hours, 0.9) FILTER (WHERE is_stage_valid) AS p90_hours
FROM stage_rows
GROUP BY delivery_outcome, stage_name;

CREATE OR REPLACE TABLE analysis.fulfillment_stage_share AS
WITH eligible AS (
    SELECT
        approval_lead_time_hours,
        precarrier_handover_time_hours,
        carrier_delivery_time_hours,
        end_to_end_fulfillment_time_hours
    FROM marts.mart_order_fulfillment
    WHERE is_primary_analysis_period
      AND is_approval_duration_valid
      AND is_precarrier_duration_valid
      AND is_carrier_duration_valid
      AND is_end_to_end_duration_valid
      AND end_to_end_fulfillment_time_hours > 0
), shares AS (
    SELECT
        'Purchase to approval' AS stage_name,
        approval_lead_time_hours / end_to_end_fulfillment_time_hours AS stage_share
    FROM eligible
    UNION ALL
    SELECT
        'Approval to carrier handover',
        precarrier_handover_time_hours / end_to_end_fulfillment_time_hours
    FROM eligible
    UNION ALL
    SELECT
        'Carrier handover to customer delivery',
        carrier_delivery_time_hours / end_to_end_fulfillment_time_hours
    FROM eligible
)
SELECT
    stage_name,
    COUNT(*) AS eligible_order_count,
    AVG(stage_share) AS average_stage_share,
    QUANTILE_CONT(stage_share, 0.5) AS p50_stage_share,
    QUANTILE_CONT(stage_share, 0.9) AS p90_stage_share
FROM shares
GROUP BY stage_name;
"""

### Regional performance and priority sellers

In [ ]:
analysis_sql["regions"] = r"""
CREATE OR REPLACE TABLE analysis.agg_customer_state_performance AS
SELECT
    customer_state,
    COUNT(*) AS total_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS delivery_eligible_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)
        AS late_delivery_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE is_delivery_sla_eligible), 0) AS late_delivery_rate,
    COUNT(*) FILTER (WHERE has_valid_review) AS reviewed_order_count,
    AVG(representative_review_score) FILTER (WHERE has_valid_review) AS average_review_score
FROM marts.mart_order_fulfillment
WHERE is_primary_analysis_period AND customer_state IS NOT NULL
GROUP BY customer_state;

CREATE OR REPLACE TABLE analysis.agg_seller_state_performance AS
SELECT
    seller_state,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible) AS handover_eligible_order_count,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)
        AS handover_breach_count,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE is_handover_sla_eligible), 0) AS handover_breach_rate,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS delivery_eligible_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)
        AS late_delivery_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE is_delivery_sla_eligible), 0) AS late_delivery_rate
FROM marts.mart_order_seller
WHERE is_primary_analysis_period
  AND seller_attribution_reliable
  AND seller_state IS NOT NULL
GROUP BY seller_state;

CREATE OR REPLACE TABLE analysis.agg_seller_customer_state_lane AS
SELECT
    seller_id,
    seller_state,
    customer_state,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible) AS handover_eligible_order_count,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)
        AS handover_breach_count,
    COUNT(*) FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE is_handover_sla_eligible), 0) AS handover_breach_rate,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible) AS delivery_eligible_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)
        AS late_delivery_order_count,
    COUNT(*) FILTER (WHERE is_delivery_sla_eligible AND NOT on_time_delivery)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE is_delivery_sla_eligible), 0) AS late_delivery_rate,
    COUNT(*) FILTER (WHERE has_valid_review) AS reviewed_order_count,
    COUNT(*) FILTER (WHERE has_valid_review AND representative_review_score <= 2)
        AS low_review_order_count,
    COUNT(*) FILTER (WHERE has_valid_review AND representative_review_score <= 2)::DOUBLE
        / NULLIF(COUNT(*) FILTER (WHERE has_valid_review), 0) AS low_review_rate,
    QUANTILE_CONT(maximum_positive_handover_delay_hours, 0.9)
        FILTER (WHERE is_handover_sla_eligible AND handover_sla_breach)
        AS breach_delay_p90_hours
FROM marts.mart_order_seller
WHERE is_primary_analysis_period
  AND seller_attribution_reliable
  AND seller_state IS NOT NULL
  AND customer_state IS NOT NULL
GROUP BY seller_id, seller_state, customer_state;

CREATE OR REPLACE TABLE analysis.mart_seller_region_investigation_queue AS
WITH classified AS (
    SELECT
        segment.*,
        seller_queue.priority_tier AS seller_priority_tier,
        seller_queue.priority_sort AS seller_priority_sort,
        seller.handover_breach_rate AS seller_handover_breach_rate,
        lane.handover_breach_rate AS state_lane_handover_breach_rate,
        lane.late_delivery_rate AS state_lane_late_delivery_rate,
        segment.handover_breach_rate
            >= seller.handover_breach_rate + parameters.monthly_rate_gap
            AS above_seller_breach_rate,
        segment.handover_breach_rate
            >= lane.handover_breach_rate + parameters.monthly_rate_gap
            AS above_state_lane_breach_rate,
        COALESCE(
            segment.late_delivery_rate
                >= lane.late_delivery_rate + parameters.customer_warning_gap,
            false
        ) AS above_state_lane_late_rate,
        parameters.minimum_seller_month_orders AS minimum_segment_orders
    FROM analysis.agg_seller_customer_state_lane AS segment
    INNER JOIN analysis.mart_seller_investigation_queue AS seller_queue USING (seller_id)
    INNER JOIN analysis.agg_seller_performance AS seller USING (seller_id)
    LEFT JOIN analysis.agg_state_lane_performance AS lane
        ON lane.seller_state = segment.seller_state
       AND lane.customer_state = segment.customer_state
    CROSS JOIN analysis.queue_parameters AS parameters
    WHERE segment.handover_eligible_order_count >= parameters.minimum_seller_month_orders
      AND segment.handover_breach_count > 0
)
SELECT
    seller_id,
    seller_state,
    customer_state,
    seller_priority_tier,
    seller_priority_sort,
    ROW_NUMBER() OVER (
        PARTITION BY seller_id
        ORDER BY handover_breach_count DESC, handover_breach_rate DESC, customer_state
    ) AS segment_rank_within_seller,
    handover_eligible_order_count,
    handover_breach_count,
    handover_breach_rate,
    seller_handover_breach_rate,
    state_lane_handover_breach_rate,
    delivery_eligible_order_count,
    late_delivery_order_count,
    late_delivery_rate,
    state_lane_late_delivery_rate,
    reviewed_order_count,
    low_review_order_count,
    low_review_rate,
    breach_delay_p90_hours,
    CONCAT_WS(
        '|',
        'SUFFICIENT_REGION_VOLUME',
        CASE WHEN above_seller_breach_rate THEN 'ABOVE_SELLER_BREACH_RATE' END,
        CASE WHEN above_state_lane_breach_rate THEN 'ABOVE_STATE_LANE_BREACH_RATE' END,
        CASE WHEN above_state_lane_late_rate THEN 'CUSTOMER_OUTCOME_WARNING' END
    ) AS reason_codes
FROM classified;
"""

### Combine analysis queries

In [ ]:
SQL_SCRIPTS["40_analysis"] = "\n\n".join(analysis_sql.values())